# AG News - BERT Fine-Tuning

## Objective

This notebook trains BERT for AG News text classification.

Two approaches are evaluated:

- Full Fine-Tuning: all BERT parameters are trainable.
- Feature-Based: BERT parameters are frozen and only the classification head is trained.

The base model is `bert-base-uncased`.

A fixed random seed of 42 is used for reproducibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')[ ]

Mounted at /content/drive


## 1. Reproducibility and Configuration

The experiments use a fixed random seed of 42.

Google Drive is mounted to store the trained models, checkpoints, and project files.

The base model used in this experiment is `bert-base-uncased`.

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/BERT_FineTuning_Project"

os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results", exist_ok=True)

print("Carpetas listas.")

Carpetas listas.


In [ ]:
!pip install -q -U transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.6 MB/s eta 0:00:00


In [ ]:
SEED = 42

import os
import random
import numpy as np
import torch
from transformers import set_seed

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

print("Seed configurada:", SEED)

Seed configurada: 42


## 2. Dataset

The AG News dataset is loaded from Hugging Face using `fancyzhx/ag_news`.

The dataset contains four news categories:

- World
- Sports
- Business
- Sci/Tech

The dataset is used for a four-class text classification task.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [ ]:
label_names = dataset["train"].features["label"].names

print("Labels:", label_names)

Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


In [ ]:
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

print(label2id)
print(id2label)

{'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}
{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}


## 3. Tokenization

The BERT tokenizer is used to convert the news text into tokens that can be processed by the model.

The maximum sequence length is 128 tokens.

The same tokenization procedure is used for both Full Fine-Tuning and Feature-Based training.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer cargado correctamente.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer cargado correctamente.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})


In [ ]:
tokenized_dataset = tokenized_dataset.rename_column(
    "label",
    "labels"
)

tokenized_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})


## 4. Full Fine-Tuning

In the Full Fine-Tuning approach, all parameters of the pretrained BERT model are trainable.

The model is adapted to the AG News classification task.

The learning rate used for this experiment is `2e-5`.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print("Modelo Full Fine-tuning cargado.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo Full Fine-tuning cargado.


In [ ]:
total_params = sum(
    p.numel()
    for p in model_full.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_full.parameters()
    if p.requires_grad
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)

Parámetros totales: 109485316
Parámetros entrenables: 109485316


In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

print("Métricas preparadas.")

Métricas preparadas.


In [ ]:
from transformers import TrainingArguments

training_args_full = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/agnews_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

print("Configuración Full preparada.")

Configuración Full preparada.


In [ ]:
from transformers import Trainer

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer Full preparado.")

Trainer Full preparado.


In [ ]:
trainer_full.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.192030,0.180871,0.942895,0.943012
2,0.115034,0.195486,0.947895,0.947896
3,0.093669,0.234656,0.948026,0.948026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=22500, training_loss=0.1497724701139662, metrics={'train_runtime': 9319.9105, 'train_samples_per_second': 38.627, 'train_steps_per_second': 2.414, 'total_flos': 2.368042020864e+16, 'train_loss': 0.1497724701139662, 'epoch': 3.0})

In [ ]:
results_full = trainer_full.evaluate()

print("AG News — Full Fine-tuning")
print("Accuracy:", results_full["eval_accuracy"])
print("F1:", results_full["eval_f1"])

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.093669,0.234656,3,0.948026,0.948026


AG News — Full Fine-tuning
Accuracy: 0.9480263157894737
F1: 0.9480263416206413


In [ ]:
MODEL_DIR_FULL = f"{BASE_DIR}/models/agnews_bert_full"

trainer_full.save_model(MODEL_DIR_FULL)
tokenizer.save_pretrained(MODEL_DIR_FULL)

print("Modelo Full guardado en:")
print(MODEL_DIR_FULL)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo Full guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/agnews_bert_full


In [ ]:
!ls -lah "$BASE_DIR/models/agnews_bert_full"

total 419M
drwx------ 2 root root 4.0K Sep 25 19:12 .
drwx------ 3 root root 4.0K Sep 25 19:12 ..
-rw------- 1 root root 1006 Sep 25 19:12 config.json
-rw------- 1 root root 418M Sep 25 19:12 model.safetensors
-rw------- 1 root root  351 Sep 25 19:12 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 19:12 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 19:12 training_args.bin


## 5. Feature-Based Training

In the Feature-Based approach, the pretrained BERT parameters are frozen.

BERT is used as a fixed feature extractor, and only the task-specific classification head is trained.

The learning rate used for this experiment is `1e-3`.

In [ ]:
model_feature = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print("Modelo Feature-based cargado.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo Feature-based cargado.


In [ ]:
for param in model_feature.base_model.parameters():
    param.requires_grad = False

for param in model_feature.classifier.parameters():
    param.requires_grad = True

total_params = sum(
    p.numel()
    for p in model_feature.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model_feature.parameters()
    if p.requires_grad
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)
print("BERT congelado correctamente.")

Parámetros totales: 109485316
Parámetros entrenables: 3076
BERT congelado correctamente.


In [ ]:
training_args_feature = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints/agnews_feature",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

print("Configuración Feature-based preparada.")

Configuración Feature-based preparada.


In [ ]:
trainer_feature = Trainer(
    model=model_feature,
    args=training_args_feature,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer Feature-based preparado.")

Trainer Feature-based preparado.


In [ ]:
trainer_feature.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.536707,0.472535,0.838026,0.836960
2,0.535511,0.439970,0.845921,0.845250
3,0.487442,0.432036,0.847368,0.847003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=22500, training_loss=0.5591378868950738, metrics={'train_runtime': 3720.4529, 'train_samples_per_second': 96.762, 'train_steps_per_second': 6.048, 'total_flos': 2.368042020864e+16, 'train_loss': 0.5591378868950738, 'epoch': 3.0})

## 6. Evaluation

The AG News models are evaluated using classification accuracy and F1 score.

The final metrics are obtained after the three training epochs.


In [ ]:
results_feature = trainer_feature.evaluate()

print("AG News — Feature-based")
print("Accuracy:", results_feature["eval_accuracy"])
print("F1:", results_feature["eval_f1"])

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.487442,0.432036,3,0.847368,0.847003


AG News — Feature-based
Accuracy: 0.8473684210526315
F1: 0.8470025010180878


In [ ]:
MODEL_DIR_FEATURE = f"{BASE_DIR}/models/agnews_bert_feature"

trainer_feature.save_model(MODEL_DIR_FEATURE)
tokenizer.save_pretrained(MODEL_DIR_FEATURE)

print("Modelo Feature-based guardado en:")
print(MODEL_DIR_FEATURE)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo Feature-based guardado en:
/content/drive/MyDrive/BERT_FineTuning_Project/models/agnews_bert_feature


In [ ]:
!ls -lah "$BASE_DIR/models/agnews_bert_feature"

total 419M
drwx------ 2 root root 4.0K Sep 25 20:17 .
drwx------ 4 root root 4.0K Sep 25 20:17 ..
-rw------- 1 root root 1006 Sep 25 20:17 config.json
-rw------- 1 root root 418M Sep 25 20:17 model.safetensors
-rw------- 1 root root  351 Sep 25 20:17 tokenizer_config.json
-rw------- 1 root root 695K Sep 25 20:17 tokenizer.json
-rw------- 1 root root 5.2K Sep 25 20:17 training_args.bin


In [ ]:
import json

results_agnews = {
    "full_fine_tuning": {
        "accuracy": results_full["eval_accuracy"],
        "f1": results_full["eval_f1"]
    },
    "feature_based": {
        "accuracy": results_feature["eval_accuracy"],
        "f1": results_feature["eval_f1"]
    }
}

results_path = f"{BASE_DIR}/results/agnews_results.json"

with open(results_path, "w") as f:
    json.dump(results_agnews, f, indent=4)

print("Resultados guardados en:")
print(results_path)

Resultados guardados en:
/content/drive/MyDrive/BERT_FineTuning_Project/results/agnews_results.json


## 7. Final Results

The Full Fine-Tuning model achieved an accuracy of 94.80% and an F1 score of 94.80%.

The Feature-Based model achieved an accuracy of 84.74% and an F1 score of 84.70%.

These results show the difference between adapting all BERT parameters and using BERT as a frozen feature extractor.